# M4 — radiation dose regression (Marswalk)

Trains a surrogate of the C4 dose model `radiation.dose_usv` — dose (uSv) from (duration, shielding, surface rate, attenuation). The dose model is a **placeholder** and its rate/attenuation await **cited RAD figures**; the regression is faithful to that model, learned in log space.

**How to run:** **Run all**, then send back **`dose_regression_report.json`** and **`dose_regression_samples.png`**.

In [ ]:
!pip install -q "git+https://github.com/MONSTER4REX/Martian-Map.git"
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', dev)

## Build data from the dose model, train, evaluate on a fresh draw

In [ ]:
import json, time
import numpy as np
from marswalk.ml.dose_regression import build_dataset, train_mlp, evaluate

Xtr, ytr = build_dataset(5000, seed=0)
Xte, yte = build_dataset(1000, seed=123)
t0 = time.time()
model = train_mlp(Xtr, ytr, epochs=200, device=dev)
rep = evaluate(model, Xte, yte)
secs = time.time() - t0
print(rep)

In [ ]:
report = {**rep.as_dict(), 'device': (torch.cuda.get_device_name(0) if dev=='cuda' else 'CPU'),
          'epochs': 200, 'seconds': round(secs, 1)}
with open('dose_regression_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
pred = model.predict(Xte)
fig, ax = plt.subplots(figsize=(5.2, 5))
ax.scatter(yte, pred, s=5, alpha=.3, color='#e2572b')
lo, hi = max(yte.min(), 1e-2), yte.max()
ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('physics dose (uSv)'); ax.set_ylabel('predicted dose (uSv)')
ax.set_title(f"dose surrogate — median rel err {rep.median_rel_err*100:.1f}%")
fig.tight_layout(); fig.savefig('dose_regression_samples.png', dpi=120)
import os; print('SAVED (download these):', os.path.abspath('dose_regression_samples.png'), '|', os.path.abspath('dose_regression_report.json'))

## Send back

Download **`dose_regression_report.json`** and **`dose_regression_samples.png`** and send them to the lead. Thanks!